# Exploratory Data Analysis — French Day-Ahead Power Prices
**EDHEC MSc Data Analysis & AI — Power Price Forecasting Thesis**

This notebook explores the ENTSO-E day-ahead price series for France (2018–2024):
- Price distribution and descriptive statistics
- Time series visualisation
- Seasonality (hourly, daily, monthly)
- Correlation with load and generation mix

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.config import INTERIM_DIR, FIGURES_DIR

sns.set_theme(style='whitegrid', palette='muted')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INTERIM_DIR / 'merged.parquet')
print(f'{len(df):,} rows | {df.shape[1]} columns | {df.index.min()} → {df.index.max()}')
df.head()

## 1. Descriptive Statistics

In [ ]:
price = df['price_da_eur_mwh']

stats = pd.DataFrame({
    'Mean (€/MWh)':   [price.mean()],
    'Median':         [price.median()],
    'Std':            [price.std()],
    'Min':            [price.min()],
    'Max':            [price.max()],
    'Negative hours': [(price < 0).sum()],
    '% negative':     [(price < 0).mean() * 100],
    '>200 €/MWh':     [(price > 200).sum()],
}).T.rename(columns={0: 'Value'})

stats['Value'] = stats['Value'].round(2)
stats

## 2. Full Price Series (2018–2024)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(price.index, price.values, linewidth=0.5, color='steelblue', alpha=0.8)
ax.axhline(0, color='red', linewidth=0.8, linestyle='--', label='0 €/MWh')
ax.set_title('French Day-Ahead Power Prices 2018–2024', fontsize=14)
ax.set_ylabel('€/MWh')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'price_series_full.png', dpi=150)
plt.show()

## 3. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(price.clip(-200, 500), bins=100, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(price.mean(), color='red', linestyle='--', label=f'Mean: {price.mean():.1f}')
axes[0].axvline(price.median(), color='orange', linestyle='--', label=f'Median: {price.median():.1f}')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('€/MWh')
axes[0].legend()

# Annual boxplot
df_plot = df[['price_da_eur_mwh']].copy()
df_plot['year'] = df_plot.index.year
df_plot.boxplot(column='price_da_eur_mwh', by='year', ax=axes[1],
                flierprops=dict(marker='.', markersize=1, alpha=0.3))
axes[1].set_title('Price Distribution by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('€/MWh')
plt.suptitle('')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'price_distribution.png', dpi=150)
plt.show()

## 4. Seasonality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
days   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

# By month
price.groupby(price.index.month).mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Avg Price by Month')
axes[0].set_xticklabels(months, rotation=45)
axes[0].set_ylabel('€/MWh')

# By day of week
price.groupby(price.index.dayofweek).mean().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Avg Price by Day of Week')
axes[1].set_xticklabels(days, rotation=45)

# By hour
price.groupby(price.index.hour).mean().plot(kind='line', ax=axes[2], color='steelblue', marker='o', markersize=4)
axes[2].set_title('Avg Price by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_ylabel('€/MWh')
axes[2].set_xticks(range(0, 24, 2))

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'price_seasonality.png', dpi=150)
plt.show()

## 5. Heatmap — Average Price by Hour × Month

In [ ]:
pivot = df['price_da_eur_mwh'].groupby(
    [df.index.month, df.index.hour]
).mean().unstack(level=0)

pivot.columns = months

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, cmap='RdYlGn_r', ax=ax, fmt='.0f', annot=False,
            cbar_kws={'label': '€/MWh'})
ax.set_title('Average Day-Ahead Price — Hour × Month (€/MWh)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Hour of Day')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'price_heatmap_hour_month.png', dpi=150)
plt.show()

## 6. Correlation — Price vs Load & Generation Mix

In [ ]:
cols = ['price_da_eur_mwh', 'load_forecast_mw', 'gen_nuclear_mw',
        'gen_wind_onshore_mw', 'gen_solar_mw', 'gen_gas_mw']
cols_available = [c for c in cols if c in df.columns]

corr = df[cols_available].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix — Price & Fundamentals', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_matrix.png', dpi=150)
plt.show()

## 7. Price vs Nuclear Generation

In [ ]:
if 'gen_nuclear_mw' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

    monthly = df[['price_da_eur_mwh', 'gen_nuclear_mw']].resample('ME').mean()

    axes[0].plot(monthly.index, monthly['price_da_eur_mwh'], color='steelblue')
    axes[0].set_ylabel('Avg Price (€/MWh)')
    axes[0].set_title('Monthly Avg Price vs Nuclear Generation')

    axes[1].plot(monthly.index, monthly['gen_nuclear_mw'] / 1000, color='orange')
    axes[1].set_ylabel('Nuclear Output (GW)')
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'price_vs_nuclear.png', dpi=150)
    plt.show()

## 8. Negative Prices Analysis

In [ ]:
neg = df[df['price_da_eur_mwh'] < 0].copy()
print(f'Total negative price hours: {len(neg):,} ({len(neg)/len(df)*100:.1f}% of dataset)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

neg.groupby(neg.index.year).size().plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Negative Price Hours per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Hours')

neg.groupby(neg.index.hour).size().plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Negative Price Hours by Hour of Day')
axes[1].set_xlabel('Hour')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'negative_prices.png', dpi=150)
plt.show()